# Week 2 · Task 3 — Train a classifier

Can we predict whether someone will return to an app the next day? In this exercise, each simulated session has three features: time spent, screens visited, and taps. The label is 1 if the simulated user returned and 0 otherwise.

From the project root, launch JupyterLab with `uv run --locked --with torch --with matplotlib jupyter lab`. Open this notebook in that environment and run its cells in order.

## 1. Create examples and split them

`X` is a table with one row per session and three columns. The numbers represent standardized features, so negative values mean below average. `y` has one label per row. A rule with some random noise creates the labels. For example, a session with features `[1, 0, 0]` and zero noise gets a score of 1.5 from the rule, so its label is 1.

Shuffle the row indices once. The model learns from 210 training sessions. We check progress on 45 separate validation sessions. We reserve 45 test sessions for the final model after comparing settings in Task 4.

In [ ]:
import torch
from torch import nn

rng = torch.Generator().manual_seed(42)

X = torch.randn(300, 3, generator=rng)
noise = 0.3 * torch.randn(300, generator=rng)
y = (1.5 * X[:, 0] + X[:, 1] - 0.5 * X[:, 2] + noise > 0).float()

train, val, test = torch.randperm(len(X), generator=rng).split([210, 45, 45])

print("Features:", X.shape, "| Labels:", y.shape)
print(f"Train: {len(train)} | Validation: {len(val)} | Test: {len(test)}")

## 2. Teach the model with the training sessions

`nn.Linear(3, 1)` uses three trainable weights and one bias to produce one raw score per session. The model starts with random parameters and learns from the feature rows and labels.

Each epoch processes the whole training split and takes one optimizer step:

1. Clear the gradients from the previous epoch.
2. Run the forward pass and calculate the loss with `BCEWithLogitsLoss`.
3. Use `loss.backward()` to compute each parameter's gradient.
4. Use `optimizer.step()` to update the parameters. SGD subtracts the learning rate times the gradient, just like the single weight update in Task 2.

We report training and validation loss after the same update, without computing gradients for either measurement.

In [ ]:
torch.manual_seed(42)

model = nn.Linear(3, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(1, 101):
    optimizer.zero_grad()

    # Remove the final dimension so scores and labels both have shape (210,).
    scores = model(X[train]).squeeze(-1)
    loss = loss_fn(scores, y[train])
    loss.backward()
    optimizer.step()

    if epoch in (1, 25, 50, 100):
        with torch.no_grad():
            train_loss = loss_fn(model(X[train]).squeeze(-1), y[train])
            val_loss = loss_fn(model(X[val]).squeeze(-1), y[val])

        print(
            f"Epoch {epoch:3d} | "
            f"Train loss: {train_loss.item():.3f} | "
            f"Validation loss: {val_loss.item():.3f}"
        )

## 3. Compare with a simple baseline

A raw score of 0 corresponds to a 50% probability after applying the sigmoid. Predict that the user returns when the score is 0 or higher.

The baseline always predicts the most common label in the training split. Score both methods on the same validation sessions, then compare their accuracy.

In [ ]:
with torch.no_grad():
    val_scores = model(X[val]).squeeze(-1)
    predictions = val_scores >= 0
    val_labels = y[val].bool()
    model_accuracy = (predictions == val_labels).float().mean()

    majority_label = y[train].mean() >= 0.5
    baseline_accuracy = (majority_label == val_labels).float().mean()

print(f"Validation accuracy: {model_accuracy.item():.1%}")
print(f"Baseline accuracy:   {baseline_accuracy.item():.1%}")

## What to record

- Compare the training loss at epochs 1 and 100.
- Record the validation accuracy for the classifier and the baseline.
- In Task 4, reuse the same split and the same parameter initialization when you compare learning rates.

## Week 2 · Task 4 — Compare learning rates

Train the same classifier for 100 epochs with learning rates of 0.01, 0.1, and 1.0. Each run uses the same train and validation splits and starts with the same weights. Record both losses after each optimizer step.

Select the learning rate with the lowest validation loss at epoch 100. The test set is reserved for the final evaluation.

In [ ]:
def compare_run(learning_rate):
    torch.manual_seed(42)
    candidate = nn.Linear(3, 1)
    candidate_optimizer = torch.optim.SGD(candidate.parameters(), lr=learning_rate)
    train_losses, val_losses = [], []

    for _ in range(100):
        candidate_optimizer.zero_grad()
        train_scores = candidate(X[train]).squeeze(-1)
        loss = loss_fn(train_scores, y[train])
        loss.backward()
        candidate_optimizer.step()

        # Measure both losses with the same updated weights.
        with torch.no_grad():
            train_scores = candidate(X[train]).squeeze(-1)
            val_scores = candidate(X[val]).squeeze(-1)
            train_losses.append(loss_fn(train_scores, y[train]).item())
            val_losses.append(loss_fn(val_scores, y[val]).item())

    return train_losses, val_losses


results = {rate: compare_run(rate) for rate in (0.01, 0.1, 1.0)}

In [ ]:
import matplotlib.pyplot as plt

fig, (train_ax, val_ax) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
epochs = range(1, 101)

for rate, (train_losses, val_losses) in results.items():
    train_ax.plot(epochs, train_losses, label=f"lr={rate}")
    val_ax.plot(epochs, val_losses, label=f"lr={rate}")
    print(
        f"lr={rate:g} | Epoch 100 train loss: {train_losses[-1]:.3f} | "
        f"validation loss: {val_losses[-1]:.3f}"
    )

for ax, title in [(train_ax, "Training loss"), (val_ax, "Validation loss")]:
    ax.set(title=title, xlabel="Epoch", ylabel="Loss")
    ax.legend()

fig.tight_layout()
plt.show()

best_rate = min(results, key=lambda rate: results[rate][1][-1])
print(f"Lowest validation loss at epoch 100: lr={best_rate:g}")

Which rate learns fastest in the first few epochs? Which finishes with the lowest validation loss? If training loss keeps falling while validation loss rises, the model is getting better on the training sessions and worse on unseen sessions. In Task 5, use the selected rate to produce a checkpoint.

## Week 2 · Task 5 — Save and reload a checkpoint

Use the learning rate selected from validation loss in Task 4. Start from the same random weights, train for 100 epochs, and save both the learned model parameters and the optimizer settings. Run all previous cells first so `X`, `y`, `train`, `val`, `loss_fn`, and `best_rate` exist.

The checkpoint file is written to the notebook's current working directory. Keep it outside Git if you are just practicing.


In [ ]:
from pathlib import Path

torch.manual_seed(42)
chosen_model = nn.Linear(3, 1)
chosen_optimizer = torch.optim.SGD(chosen_model.parameters(), lr=best_rate)

for epoch in range(1, 101):
    chosen_optimizer.zero_grad()
    scores = chosen_model(X[train]).squeeze(-1)
    loss = loss_fn(scores, y[train])
    loss.backward()
    chosen_optimizer.step()

checkpoint_path = Path("week02_classifier.pt")
torch.save(
    {
        "epoch": epoch,
        "learning_rate": best_rate,
        "model_state": chosen_model.state_dict(),
        "optimizer_state": chosen_optimizer.state_dict(),
    },
    checkpoint_path,
)
print("Saved:", checkpoint_path.resolve())

## Reload into a new model

A fresh `nn.Linear(3, 1)` starts with different weights. Loading the saved `model_state` replaces them with your trained weights. The optimizer's state restores the settings used to train it.

This exercise uses SGD without momentum. Its optimizer state contains the learning rate and parameter groups, with no running momentum values.


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

reloaded_model = nn.Linear(3, 1)
reloaded_optimizer = torch.optim.SGD(
    reloaded_model.parameters(), lr=checkpoint["learning_rate"]
)
reloaded_model.load_state_dict(checkpoint["model_state"])
reloaded_optimizer.load_state_dict(checkpoint["optimizer_state"])

with torch.no_grad():
    original_scores = chosen_model(X[val])
    reloaded_scores = reloaded_model(X[val])

torch.testing.assert_close(original_scores, reloaded_scores)
assert reloaded_optimizer.param_groups[0]["lr"] == best_rate

print("Reloaded epoch:", checkpoint["epoch"])
print("Reloaded learning rate:", reloaded_optimizer.param_groups[0]["lr"])
print("Validation predictions match after reload.")